# Vectorizaciones nuevas en Google Colab

Monta Google Drive, usa el repositorio de Drive y ejecuta solo `robertuito-base-uncased`, `beto-raw` y `robertalex` (sustituto disponible de RoBERTa-BNE), para mensajes y usuarios con `mean pooling`. No ejecuta `run_vectorizations.sh` completo.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

from google.colab import drive

drive.mount('/content/drive')

# Cambia esta ruta si el repositorio tiene otro nombre o está en otra carpeta.
DRIVE_REPO_PATH = '/content/drive/MyDrive/dm_project'
REPO = Path(DRIVE_REPO_PATH)

if not REPO.is_dir():
    raise FileNotFoundError(
        f'No existe {REPO}. Sube o clona dm_project en Google Drive y '
        'actualiza DRIVE_REPO_PATH.'
    )
if not (REPO / 'vectorization').is_dir():
    raise FileNotFoundError(
        f'{REPO} no parece ser la raíz del repositorio: falta vectorization/'
    )

os.chdir(REPO)
print(f'Repositorio seleccionado: {REPO}')

# Ruta del corpus. Si está fuera del repositorio, sustituye este valor.
CORPUS_ROOT = REPO / 'MentalRiskEs2023' / 'corpus'
if not CORPUS_ROOT.is_dir():
    raise FileNotFoundError(
        f'No existe el corpus en {CORPUS_ROOT}. '
        'Configura CORPUS_ROOT con la carpeta que contiene Ansiedad, '
        'Depresion y TCA.'
    )
corpus_disorders = [CORPUS_ROOT / name for name in ('Ansiedad', 'Depresion', 'TCA')]
print(f'Corpus seleccionado: {CORPUS_ROOT}')
print('Carpetas encontradas:', [path.name for path in corpus_disorders if path.is_dir()])

## Dependencias

In [ ]:
requirements = REPO / 'vectorization' / 'requirements.txt'
if not requirements.is_file():
    raise FileNotFoundError(f'No existe {requirements}')

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(requirements)],
    check=True,
)
subprocess.run(
    [sys.executable, '-m', 'vectorization', 'transformer', '--help'],
    cwd=REPO,
    check=True,
)

In [ ]:
import torch

print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Ejecutar las seis vectorizaciones

Las salidas se guardan en `vectorizations_colab/` dentro de Drive. Si un modelo falla, se continúa con los siguientes y se muestra el estado final.

In [ ]:
OUTPUT_DIR = REPO / 'vectorizations_colab'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

models = [
    'robertuito-base-uncased',
    'beto-raw',
    'robertalex',
]
experiments = [
    (model, unit)
    for model in models
    for unit in ('message', 'user')
]
results = []

for model, unit in experiments:
    command = [
        sys.executable, '-m', 'vectorization', 'transformer',
        '--model', model,
        '--unit', unit,
        '--corpus-root', str(CORPUS_ROOT),
        '--output-dir', str(OUTPUT_DIR),
        '--overwrite',
    ]
    if unit == 'user':
        command.extend(['--user-pooling', 'mean'])
    print('\n' + '=' * 80)
    print('$ ' + ' '.join(command))
    completed = subprocess.run(
        command,
        cwd=REPO,
        capture_output=True,
        text=True,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print('--- stderr ---')
        print(completed.stderr)
    status = 'OK' if completed.returncode == 0 else f'FALLO ({completed.returncode})'
    results.append({'model': model, 'unit': unit, 'status': status})
    print(f'Resultado: {status}')

print('\nResumen')
for result in results:
    print(f"- {result['model']} / {result['unit']}: {result['status']}")
print(f'\nSalidas: {OUTPUT_DIR}')